# 02 · FinSight pipeline: TAT-LLM + executor + self-consistency (main results)

**No training on our side.** Stage 2 uses TAT-LLM (Zhu et al., 2024), LLaMA-2 already fine-tuned by its authors on FinQA, TAT-QA and TAT-DQA (`next-tat/tat-llm-7b-fft`, public). Its prompt is rebuilt exactly as in the authors' released files.

**Our additions, each measured separately with the official, unit-aware scorer:**
1. **Python executor**: the model's equation is executed exactly instead of trusting its own arithmetic, with x100 reconciliation for percentages.
2. **Self-consistency voting**: K sampled answers + the greedy answer are executed; the most common result wins.
3. **Evidence grounding**: the model's evidence values are located in the page rows (evidence recall / precision).

Part A uses the same 200 dev questions as notebook 01. Part C is the single final run on the official test set.

Rough T4 time: Part A ≈ 30 min (model download ≈ 10 min the first time), Part B ≈ 1–1.5 h, Part C ≈ 3–4 h. Every part saves to Drive and resumes after a disconnect (re-run setup, the model cell, then the same cell).

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import json, torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def report_row(r):
    return {m: v for m, v in r.items() if m != "rows"}
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
from finsight import tatllm

MODEL_ID = "next-tat/tat-llm-7b-fft"    # or "next-tat/tat-llm-13b-fft" (slower; use batch_size=2)
name = MODEL_ID.split("/")[-1]
model, tok = tatllm.load(MODEL_ID)

## Part A · Greedy answers: model's own arithmetic vs. our executor (dev sample)

In [ ]:
N = 200
eval_qs = sample(q_dev, N)                # same sample as notebook 01
doc_of = {q["uid"]: docs_dev[q["doc_uid"]] for q in eval_qs}
raw = tatllm.predict(model, tok, docs_dev, eval_qs, f"{OUT}/{name}_dev{N}.jsonl", batch_size=4)

RESULTS = {}
base_file = f"{OUT}/zeroshot_Qwen2.5-7B-Instruct_dev{N}.jsonl"
if os.path.exists(base_file):
    base = {json.loads(l)["uid"]: json.loads(l) for l in open(base_file)}
    RESULTS["General LLM: Qwen2.5-7B, no training (notebook 01)"] = score(eval_qs, base)
own = tatllm.score_outputs(raw, use_executor=False, doc_of=doc_of)
exe = tatllm.score_outputs(raw, use_executor=True, doc_of=doc_of)
RESULTS[f"{name}: model's own arithmetic (authors' setup)"] = score(eval_qs, own)
RESULTS[f"{name} + executor (ours)"] = score(eval_qs, exe)
for label, r in RESULTS.items():
    print_report(r, label)

## Part B · Self-consistency voting (dev sample)

In [ ]:
K = 5
samples = tatllm.sample(model, tok, docs_dev, eval_qs, f"{OUT}/{name}_dev{N}_k{K}.jsonl", k=K, batch_size=2)
voted = {q["uid"]: tatllm.vote(samples[q["uid"]]["samples"], raw[q["uid"]]["raw"], doc_of[q["uid"]])
         for q in eval_qs}
RESULTS[f"{name} + executor + voting (k={K}) (ours)"] = score(eval_qs, voted)
print_report(RESULTS[f"{name} + executor + voting (k={K}) (ours)"], "with voting")

print("\nSummary (official scorer, same 200 dev questions):")
for label, r in RESULTS.items():
    print(f"  {r['EM']:6.2f} EM  {r['F1']:6.2f} F1  unit {r['scale_acc']:5.1f}  {label}")
json.dump({k: report_row(r) for k, r in RESULTS.items()}, open(f"{OUT}/finsight_dev{N}_results.json", "w"), indent=1)

In [ ]:
# Mistakes of the best setup
result, preds = RESULTS[f"{name} + executor + voting (k={K}) (ours)"], voted
# Look at mistakes: this is where the next improvement comes from
rows = {r["uid"]: r for r in result["rows"]}
wrong = [q for q in eval_qs if rows[q["uid"]]["em"] == 0]
print(len(wrong), "wrong of", len(eval_qs))
for q in wrong[:10]:
    p = preds[q["uid"]]
    print(f"[{q['answer_type']}] {q['question']}")
    print(f"   gold: {q['answer']} ({q['scale']}) | evidence rows {q['evidence']} | derivation {q['derivation']!r}")
    print(f"   pred: {p['answer']} ({p['scale']}) | evidence rows {p['evidence']}")
    print("   raw:", p["raw"].replace(chr(10), " / ")[:400], "\n")

## Part C · Final run on the official TEST set (run once, at the end)
Greedy + executor on all 1,663 test questions (≈ 3–4 h on a T4). Set `VOTE = True` to add voting (≈ 5x longer; only if you have the GPU time). Writes `test_official.json` in the official `{uid: [answer, scale]}` format.

In [ ]:
VOTE = False
doc_test = {q["uid"]: docs_test[q["doc_uid"]] for q in q_test}
raw_t = tatllm.predict(model, tok, docs_test, q_test, f"{OUT}/{name}_test.jsonl", batch_size=4)
own_t = tatllm.score_outputs(raw_t, use_executor=False, doc_of=doc_test)
final_t = tatllm.score_outputs(raw_t, use_executor=True, doc_of=doc_test)
TEST = {f"{name}: model's own arithmetic": score(q_test, own_t), f"{name} + executor (ours)": score(q_test, final_t)}
if VOTE:
    s_t = tatllm.sample(model, tok, docs_test, q_test, f"{OUT}/{name}_test_k5.jsonl", k=5, batch_size=2)
    final_t = {q["uid"]: tatllm.vote(s_t[q["uid"]]["samples"], raw_t[q["uid"]]["raw"], doc_test[q["uid"]]) for q in q_test}
    TEST[f"{name} + executor + voting (ours)"] = score(q_test, final_t)
for label, r in TEST.items():
    print_report(r, "TEST: " + label)
json.dump({u: [p["answer"], p["scale"]] for u, p in final_t.items()}, open(f"{OUT}/test_official.json", "w"))
json.dump({k: report_row(r) for k, r in TEST.items()}, open(f"{OUT}/finsight_test_results.json", "w"), indent=1)

print("\nPublished TAT-DQA test results for comparison (EM): MHST 41.5 · Doc2SoarGraph 59.2 · GPT-4 64.5 · "
      "TAT-LLM-7B 72.6 (authors' lenient evaluation script)")